In [1]:
!gdown 1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1

Downloading...
From: https://drive.google.com/uc?id=1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1
To: /content/electricity.csv
100% 95.6M/95.6M [00:01<00:00, 74.2MB/s]


In [2]:
!rm -r sample_data

In [3]:
!git clone https://github.com/Morteza-24/transformer-timeseries.git

Cloning into 'transformer-timeseries'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 316 (delta 141), reused 295 (delta 123), pack-reused 0 (from 0)
Receiving objects: 100% (316/316), 16.64 MiB | 16.56 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [4]:
!mv transformer-timeseries/* transformer-timeseries/.git/ ./
!rm -d transformer-timeseries/

In [5]:
!mkdir datasets
!mv electricity.csv datasets

In [6]:
%pip install reformer-pytorch

In [1]:
import os
import random
import sys
from argparse import Namespace
from pathlib import Path

import numpy as np
import torch

# Run from the iTransformer project root so dataset/checkpoint paths resolve correctly.
# ITRANSFORMER_ROOT = (Path.cwd().parent / "models" / "itransformer").resolve()
ITRANSFORMER_ROOT = (Path.cwd() / "models" / "itransformer").resolve()
os.chdir(ITRANSFORMER_ROOT)
sys.path.insert(0, str(ITRANSFORMER_ROOT))


In [2]:
from experiments.exp_long_term_forecasting import Exp_Long_Term_Forecast
from experiments.exp_long_term_forecasting_partial import Exp_Long_Term_Forecast_Partial

# User-specified arguments; everything else uses run.py defaults.
args = Namespace(
    # user-specified
    is_training=1,
    root_path="../../datasets/",
    data_path="electricity.csv",
    model_id="ECL_96_96",
    model="Informer",
    data="custom",
    features="M",
    seq_len=96,
    pred_len=96,
    e_layers=3,
    enc_in=321,
    dec_in=321,
    c_out=321,
    des="Exp",
    d_model=512,
    d_ff=512,
    batch_size=16,
    learning_rate=0.0005,
    itr=1,
    # defaults from run.py
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",
    label_len=48,
    n_heads=8,
    d_layers=1,
    moving_avg=25,
    factor=1,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,
    do_predict=False,
    num_workers=10,
    train_epochs=5,
    patience=3,
    loss="MSE",
    lradj="type1",
    use_amp=False,
    use_gpu=True,
    gpu=0,
    use_multi_gpu=False,
    devices="0,1,2,3",
    exp_name="MTSF",
    channel_independence=False,
    inverse=False,
    class_strategy="projection",
    target_root_path="./data/electricity/",
    target_data_path="electricity.csv",
    efficient_training=False,
    use_norm=True,
    partial_start_index=0,
)

fix_seed = 24
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(" ", "")
    device_ids = args.devices.split(",")
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

print("Args in experiment:")
print(args)

if args.exp_name == "partial_train":
    Exp = Exp_Long_Term_Forecast_Partial
else:
    Exp = Exp_Long_Term_Forecast

if args.is_training:
    for ii in range(args.itr):
        setting = "{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}".format(
            args.model_id,
            args.model,
            args.data,
            args.features,
            args.seq_len,
            args.label_len,
            args.pred_len,
            args.d_model,
            args.n_heads,
            args.e_layers,
            args.d_layers,
            args.d_ff,
            args.factor,
            args.embed,
            args.distil,
            args.des,
            args.class_strategy,
            ii,
        )

        exp = Exp(args)
        print(">>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>".format(setting))
        exp.train(setting)

        print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
        exp.test(setting)

        if args.do_predict:
            print(">>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
            exp.predict(setting, True)

        torch.cuda.empty_cache()
else:
    ii = 0
    setting = "{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}".format(
        args.model_id,
        args.model,
        args.data,
        args.features,
        args.seq_len,
        args.label_len,
        args.pred_len,
        args.d_model,
        args.n_heads,
        args.e_layers,
        args.d_layers,
        args.d_ff,
        args.factor,
        args.embed,
        args.distil,
        args.des,
        args.class_strategy,
        ii,
    )

    exp = Exp(args)
    print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
    exp.test(setting, test=1)
    torch.cuda.empty_cache()

Args in experiment:
Namespace(is_training=1, root_path='../../datasets/', data_path='electricity.csv', model_id='ECL_96_96', model='Informer', data='custom', features='M', seq_len=96, pred_len=96, e_layers=3, enc_in=321, dec_in=321, c_out=321, des='Exp', d_model=512, d_ff=512, batch_size=16, learning_rate=0.0005, itr=1, target='OT', freq='h', checkpoints='./checkpoints/', label_len=48, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, do_predict=False, num_workers=10, train_epochs=5, patience=3, loss='MSE', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0)
Use GPU: cuda:0
>>>>>>>start training : ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl

In [3]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   utils/tools.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../../.config/
	../../datasets/
	checkpoints/
	result_long_term_forecast.txt
	results/
	test_results/

no changes added to commit (use "git add" and/or "git commit -a")


In [4]:
!zip -r9 checkpoints.zip checkpoints
!zip -r9 results.zip results
!zip -r9 test_results.zip test_results

  adding: checkpoints/ (stored 0%)
  adding: checkpoints/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/ (stored 0%)
  adding: checkpoints/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/checkpoint.pth (deflated 8%)
  adding: results/ (stored 0%)
  adding: results/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/ (stored 0%)
  adding: results/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/metrics.npy (deflated 40%)
  adding: results/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/pred.npy (deflated 8%)
  adding: results/ECL_96_96_Informer_custom_M_ft96_sl48_ll96_pl512_dm8_nh3_el1_dl512_df1_fctimeF_ebTrue_dtExp_projection_0/true.npy (deflated 14%)
  adding: test_results/ (stored 0%)
  adding: test_results/